# Pesa Salama: Master Dataset Preparation for Tableau Dashboard

This notebook prepares the final analytics-ready dataset for Tableau visualization.
It combines raw Google Play Store review data with NLP outputs such as sentiment,
complaint classification, fraud indicators, and distress scores.


In [1]:
import pandas as pd
import numpy as np

In [2]:
RAW_PATH = 'MASTER_RAW_kenya_fintech.csv'
CLEAN_PATH = 'cleaned_data.csv'
OUTPUT = 'pesa_salama_MASTER_tableau.csv'

## Load Raw and Cleaned Datasets

We load:
- Raw fintech review data
- Cleaned NLP-enhanced dataset

In [3]:
raw = pd.read_csv(RAW_PATH)
clean = pd.read_csv(CLEAN_PATH)

print(raw.shape)
print(clean.shape)

(53507, 12)
(53506, 23)


## Date Feature Engineering

Create dashboard-friendly date features:
- Date
- Month-Year
- Year

In [4]:
raw['at'] = pd.to_datetime(raw['at'])

raw['date'] = raw['at'].dt.date.astype(str)
raw['month_year'] = raw['at'].dt.to_period('M').astype(str)
raw['year'] = raw['at'].dt.year.astype(int)

## Standardize App Names

Normalize app names for consistent reporting and visualization.

In [5]:
raw['app_name'] = raw['app_name'].map({
    'mpesa': 'M-Pesa',
    'mysafaricom': 'MySafaricom',
    'equity': 'Equity Mobile',
    'branch': 'Branch',
    'tala': 'Tala',
    'kcb': 'KCB Mobile'
})

## User Engagement Features

Generate engagement indicators:
- Reply status
- High engagement reviews
- Review length
- Short review flag

In [6]:
raw['has_reply'] = raw['replyContent'].notna().astype(int)

raw['high_engagement'] = (
    raw['thumbsUpCount'] > 50
).astype(int)

raw['review_length'] = (
    raw['content']
    .fillna('')
    .str.len()
)

raw['is_short_review'] = (
    raw['review_length'] < 15
).astype(int)

## Rating Classification

Group star ratings into:
- Positive
- Neutral
- Negative

In [7]:
def rate_bucket(score):
    if score <= 2:
        return 'Negative (1-2 stars)'
    elif score == 3:
        return 'Neutral (3 stars)'
    else:
        return 'Positive (4-5 stars)'


raw['rating_bucket'] = raw['score'].apply(rate_bucket)

## NLP Feature Preparation

Prepare:
- Sentiment labels
- Complaint labels
- Fraud indicators
- Language detection

In [8]:
clean_sub = clean[
    [
        'reviewId',
        'sentiment_label',
        'complaint_label',
        'fraud_indicator',
        'final_language'
    ]
].copy()

clean_sub['sentiment_label'] = (
    clean_sub['sentiment_label']
    .str.capitalize()
)

clean_sub['complaint_label'] = (
    clean_sub['complaint_label']
    .str.replace('_', ' ')
    .str.title()
)

clean_sub['sentiment_score'] = (
    clean_sub['sentiment_label']
    .map({
        'Positive': 1,
        'Neutral': 0,
        'Negative': -1
    })
)

## Merge Raw and NLP Datasets

In [9]:
master = raw.merge(
    clean_sub,
    on='reviewId',
    how='left'
)

master.head()

,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,...,has_reply,high_engagement,review_length,is_short_review,rating_bucket,sentiment_label,complaint_label,fraud_indicator,final_language,sentiment_score
0,695de54e-4b85-4669-ae8c-ad2fdf16667e,A Google user,https://play-lh.googleusercontent.com/EGemoI2N...,"The app still has issues on OTP, because I hav...",1,0,5.1.7,2026-05-11 11:38:40,NaN,NaN,...,0,0,311,0,Negative (1-2 stars),Negative,App Issue,0.0,english,-1.0
1,acd5c061-de13-474b-8645-f628044f2a50,A Google user,https://play-lh.googleusercontent.com/EGemoI2N...,si everytime nitakuwa na bundles za ku check m...,2,0,5.1.1,2026-05-11 11:22:24,NaN,NaN,...,0,0,196,0,Negative (1-2 stars),Negative,General,0.0,english,-1.0
2,6f9f52e9-0a00-4f70-a1cc-7687f28465a3,A Google user,https://play-lh.googleusercontent.com/EGemoI2N...,this is the stupidest app ever from saf. the w...,1,0,5.1.7,2026-05-11 11:16:47,NaN,NaN,...,0,0,50,0,Negative (1-2 stars),Negative,General,0.0,english,-1.0
3,4a605b22-efc1-4641-b79e-e166b4a7b2e4,A Google user,https://play-lh.googleusercontent.com/EGemoI2N...,Life must go on without this useless app. It u...,1,0,1.14.2,2026-05-11 11:01:23,NaN,NaN,...,0,0,125,0,Negative (1-2 stars),Negative,General,0.0,english,-1.0
4,fd284f23-d966-4be5-b421-a1f0e14c1e13,A Google user,https://play-lh.googleusercontent.com/EGemoI2N...,the upgrade is terrible,1,0,NaN,2026-05-11 10:45:52,NaN,NaN,...,0,0,23,0,Negative (1-2 stars),Negative,General,0.0,english,-1.0


## Financial Distress Score

The distress score combines:
- Fraud complaint rate
- Negative sentiment rate

Formula:

Distress Score = ((Fraud Rate × 0.6) + (Negative Rate × 0.4)) × 10

In [10]:
fraud_rate = (
    master.groupby(['app_name', 'month_year'])['fraud_indicator']
    .mean()
    .reset_index(name='fraud_rate')
)

neg_rate = (
    master.groupby(['app_name', 'month_year'])
    .apply(
        lambda x: (
            x['sentiment_label'] == 'Negative'
        ).mean()
    )
    .reset_index(name='neg_rate')
)

distress = fraud_rate.merge(
    neg_rate,
    on=['app_name', 'month_year']
)

distress['distress_score'] = (
    (
        distress['fraud_rate'] * 0.6 +
        distress['neg_rate'] * 0.4
    ) * 10
).round(3)

distress.head()

,app_name,month_year,fraud_rate,neg_rate,distress_score
0,Branch,2026-04,0.007899,0.138099,0.600
1,Branch,2026-05,0.012895,0.177205,0.786
2,Equity Mobile,2024-10,0.000000,0.181655,0.727
3,Equity Mobile,2024-11,0.001590,0.151033,0.614
4,Equity Mobile,2024-12,0.001529,0.117737,0.480


In [11]:
#Merge Distress Score
master = master.merge(
    distress[
        ['app_name', 'month_year', 'distress_score']
    ],
    on=['app_name', 'month_year'],
    how='left'
)

## Final Tableau Dataset

In [12]:
output = master[
    [
        'reviewId',
        'app_name',
        'score',
        'rating_bucket',
        'thumbsUpCount',
        'high_engagement',
        'has_reply',
        'date',
        'month_year',
        'year',
        'review_length',
        'is_short_review',
        'final_language',
        'sentiment_label',
        'sentiment_score',
        'complaint_label',
        'fraud_indicator',
        'distress_score'
    ]
].copy()

In [13]:
#renaming column names
output.columns = [
    'review_id',
    'app_name',
    'star_rating',
    'rating_bucket',
    'thumbs_up',
    'high_engagement',
    'has_reply',
    'date',
    'month_year',
    'year',
    'review_length',
    'is_short_review',
    'language',
    'sentiment',
    'sentiment_score',
    'complaint_type',
    'fraud_flag',
    'distress_score'
]

## Export Tableau Dataset

In [14]:

output.to_csv('pesa_salama_MASTER_tableau.csv', index=False)

In [15]:
#data validation
output.info()

output.sample(5)

<class 'pandas.DataFrame'>
RangeIndex: 53507 entries, 0 to 53506
Data columns (total 18 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   review_id        53507 non-null  str    
 1   app_name         53507 non-null  str    
 2   star_rating      53507 non-null  int64  
 3   rating_bucket    53507 non-null  str    
 4   thumbs_up        53507 non-null  int64  
 5   high_engagement  53507 non-null  int64  
 6   has_reply        53507 non-null  int64  
 7   date             53507 non-null  str    
 8   month_year       53507 non-null  str    
 9   year             53507 non-null  int64  
 10  review_length    53507 non-null  int64  
 11  is_short_review  53507 non-null  int64  
 12  language         53506 non-null  str    
 13  sentiment        53506 non-null  str    
 14  sentiment_score  53506 non-null  float64
 15  complaint_type   53506 non-null  str    
 16  fraud_flag       53506 non-null  float64
 17  distress_score   53507 

,review_id,app_name,star_rating,rating_bucket,thumbs_up,high_engagement,has_reply,date,month_year,year,review_length,is_short_review,language,sentiment,sentiment_score,complaint_type,fraud_flag,distress_score
28162,3bd63704-adb3-4997-b44a-08fc21a9c014,Equity Mobile,5,Positive (4-5 stars),0,0,1,2025-07-06,2025-07,2025,13,1,english,Positive,1.0,General,0.0,0.432
26968,4a887f7c-c90c-44c5-ae44-90efafeaa90c,Equity Mobile,5,Positive (4-5 stars),0,0,1,2025-09-14,2025-09,2025,18,0,english,Positive,1.0,General,0.0,0.593
11164,99cd8bd2-6499-44b1-a006-856346f6862e,MySafaricom,5,Positive (4-5 stars),0,0,0,2026-02-16,2026-02,2026,9,1,english,Positive,1.0,General,0.0,0.827
2630,9712b887-c82e-4e06-970a-f56980e8b427,M-Pesa,5,Positive (4-5 stars),0,0,0,2026-04-16,2026-04,2026,6,1,english,Positive,1.0,General,0.0,2.869
9775,ca16cd5d-1c86-430b-9fbb-bb31b9b91d7a,M-Pesa,5,Positive (4-5 stars),0,0,0,2025-12-25,2025-12,2025,4,1,english,Positive,1.0,General,0.0,0.324


## Tableau Dashboard

[View Interactive Dashboard](https://public.tableau.com/views/pesasalama/Dashboard?:language=en-US&publish=yes&:sid=&:redirect=auth&:display_count=n&:origin=viz_share_link)
## Features
- Sentiment Analysis
- Fraud Complaint Detection
- Distress Score Calculation
- Tableau Dashboard
- NLP Complaint Classification
- Fintech Consumer Insights